In [6]:
from typing import Any
from langchain.agents.middleware import (AgentMiddleware, AgentState, hook_config)
from langchain.agents.middleware import (PIIMiddleware, HumanInTheLoopMiddleware)
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool 
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import AIMessage

class HealthcareSafetyFilter(AgentMiddleware):
    """Block non medical or harmfull request"""
    Blocked_topic=[
        "Drug_synthesis", "self-harm", "suicide method", "weapon", "hack"
    ]
    @hook_config(can_jump_to=["end"])
    def before_agent_call(self, state: AgentState, runtime: Runtime)-> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_msg=state["messages"][0]
        if first_msg.type != "human":
            return None
        
        content= first_msg.content.lower()
        for topic in self.Blocked_topic:
            for topic in content:
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I am a healthcare assistant and can only help"
                            "with medical question, appointments and health information"
                            "if you are in crisis, please call 1122 or your local emergency number"
                        )
                    }],
                    "jump_to": "end"
                }
        return None

# Medical Output Validator

In [7]:
class MedicalOutputValidator(AgentMiddleware):
    "Ensure all responses include appropriate medical disclaimers"
    DISCLAIMER= (
        "\n\n This is general health information, not medica advice."
        "Please consult a qualified healthcare professional"
    )

    @hook_config(can_jump_to=["end"])

    def after_agent(self, state: AgentState, runtime: Runtime)-> dict[str, Any] | None:
        if not state["messages"]:
            return None
        
        last_msg=state["messages"][-1]
        if not isinstance(last_msg, AIMessage):
            return None
        
        # Add disclaimer if already not present 
        if "medical advice" not in last_msg.content.lower():
            last_msg.content+= self.DISCLAIMER
        
        return None


# HealthCare Tool 

In [8]:
@tool
def search_symptoms(symptoms: str)-> str:
    """Search for information about medical symptoms"""
    return (
        f"Symptoms information for : {symptoms}"
        "Please Consult a doctor for diagnosis")
    
@tool
def book_appointment(patient_name: str, date: str, doctor: str)-> str:
    """Book the medical appointment"""
    return (
        f"Appointment booked for {Patient_name}"
        f"With Dr. {doctor} on {date}"
    )
@tool 
def get_medication_info(medication: str)-> str:
    """Get information about a medication."""
    return (
        f"General info about {medication}."
        "Always follow your doctor prescription"
    )

# Assembling HealthCare Chatbot

In [10]:

healthcare_bot=create_agent(
    model="gpt-40-mini",
    tools=[search_symptoms, book_appointment, get_medication_info],
    middleware=[
        # Guardrail 1: Block Harmfull/off-topic request
        HealthcareSafetyFilter(),

        # Guardrail 2: Redact patient PII from inputs
        PIIMiddleware(
            "email", strategy="redact", apply_to_input=True
        ),
        PIIMiddleware(
            "credit_card", strategy="redact", apply_to_input=True
        ),

        # guardrail 3: Require approval before booking appointment
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_appointment": True,
                "search_symptoms" : False,
                "get_medication_info": False,
            }
        ),

        # Guardrail 4: Add medical disclaimer to all outputs
        MedicalOutputValidator(),
    ],
    checkpointer= InMemorySaver(),
    system_prompt=(
        "you are a helpfull healthcare assistant"
        "you can search for symptoms, medication information,"
        "and help to book appointment. Always be empathetic and"
        "remind user to consult a doctor for diagonsis"
    ),
)
print("Healthcare chatbot with full guardrail stack created!")

Healthcare chatbot with full guardrail stack created!
